# 3 · Train, evaluate and export the models

Uses the search result from notebook 2 (`data/models/hyperparameters.json`, or defaults if it does not exist yet) and

1. evaluates each model **leave-one-sausage-out**: train on four sausages, score on the fifth;
2. saves one model per fold (`data/models/folds/`). The demonstrator replays a sausage with the model that never saw it;
3. trains the final models on all annotated sausages (`data/models/final/`);
4. writes `manifest.json` and `metrics.json`, which the demonstrator's evaluation panel shows.

It also checks the models on the **unannotated series 2** (summer → spring): phases can only advance in time, so the
predicted phase should rise there too. A model that learned the season instead of the ripening runs backwards
(details and the comparison of feature sets: notebook 04).

`FEATURE_SET` must match the one used in notebook 2. `LIMIT_SIZE` caps the Random Forest (100 trees, depth 10). An unlimited forest on this data grows to hundreds of MB.
**Train with the same scikit-learn version the demo laptop runs**, because model files are not portable across versions.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import glob
import matplotlib.pyplot as plt
import pandas as pd
from ml_core import data
from ml_core.training import (AUGMENTATION, DEFAULT_FEATURE_SET, augment_temperature, build_steps, load_hyperparameters,
                              load_windows, metrics_table, predict_progression, train_and_evaluate, training_params)

In [ ]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = 10, 2
FEATURE_SET = DEFAULT_FEATURE_SET   # see FEATURE_SETS in ml_core/training.py
AUGMENT = False                     # True: augment temperature features in the training data (see notebook 04)
LIMIT_SIZE = True
FOLD_MODELS = True
QUICK = False
N_JOBS = -1          # worker processes for folds (-1 = all cores); lower it if RAM runs short

In [ ]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE)
unannotated = None
if glob.glob(os.path.join(CONVERTED_DIR, "series_2_*.csv")):
    unannotated = load_windows(CONVERTED_DIR, "series_2_*.csv", WINDOW_SIZE, STRIDE, annotated=False, classes=windows.classes)
windows.summary()

In [ ]:
metrics = train_and_evaluate(windows, MODELS_DIR, FEATURE_SET, unannotated, augmentation=AUGMENTATION if AUGMENT else None,
                             limit_size=LIMIT_SIZE, fold_models=FOLD_MODELS, quick=QUICK, n_jobs=N_JOBS)

## Results (mean ± std over the held-out sausages)

In [ ]:
metrics_table(metrics)

In [ ]:
# balanced accuracy per held-out sausage
pd.DataFrame({m["name"]: {f["held_out"]: round(f["balanced_accuracy"], 3) for f in m["folds"]}
              for m in metrics["models"].values()})

In [ ]:
# confusion matrices, row-normalised (rows: annotation, columns: prediction)
for m in metrics["models"].values():
    cm = pd.DataFrame(m["confusion_matrix"], index=metrics["classes"], columns=metrics["classes"])
    print(m["name"])
    display((cm.div(cm.sum(axis=1), axis=0) * 100).round(1))

In [ ]:
# how often two models give the same label on held-out windows
pd.Series(metrics["agreement_on_held_out"]).round(3)

## Plausibility on the unannotated series 2

Predicted phase over time (smoothed over one day). Plausible: a staircase going up. A model that follows the
temperature instead goes down in autumn and winter. ρ in the table above: +1 = phases only advance.

In [ ]:
if unannotated is not None:
    params = load_hyperparameters(MODELS_DIR)[0]
    w, u = windows.select(FEATURE_SET), unannotated.select(FEATURE_SET)
    X, y, _ = w.stack()
    if AUGMENT:
        X, y = augment_temperature(X, y)
    fig, axes = plt.subplots(1, len(metrics["models"]), figsize=(15, 3.2), sharey=True)
    for ax, key in zip(axes, metrics["models"]):
        steps = build_steps(key, training_params(key, params[key], LIMIT_SIZE, QUICK)).fit(X, y)
        for rec, s in predict_progression(steps, u, w.classes).items():
            ax.plot(s.index, s.values, label=rec, lw=1)
        ax.set(title=metrics["models"][key]["name"], yticks=range(len(w.classes)), yticklabels=w.classes)
        ax.tick_params(axis="x", rotation=30)
    axes[0].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

**Next:** start the demonstrator (`scripts/start_demo.ps1`). It loads the models from `data/models/`.
Read the scores with care: all annotated sausages come from one curing run.